# TFLite Export — ASL Sign Language Model
Standalone notebook. **No need to re-run training notebooks.**  
Loads best model (`EfficientNetB0 fine-tuned`) directly from Google Drive and exports TFLite variants for deployment on Raspberry Pi.

## 0. Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Imports & Dataset

In [2]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

# Download test split from Kaggle (for INT8 calibration)
if not os.path.exists("sign_mnist_test.csv") and not os.path.exists("sign_mnist_test/sign_mnist_test.csv"):
    os.environ['KAGGLE_USERNAME'] = "abdullahashiry"
    os.environ['KAGGLE_KEY']      = "KGAT_331632d901a6cb7a05431b55135bd8c2"
    !pip install -q kaggle
    !kaggle datasets download -d datamunge/sign-language-mnist --unzip

test_path = ("sign_mnist_test/sign_mnist_test.csv"
             if os.path.exists("sign_mnist_test/sign_mnist_test.csv")
             else "sign_mnist_test.csv")

test_df = pd.read_csv(test_path)
x_test  = test_df.drop("label", axis=1).values.reshape(-1, 28, 28, 1).astype("float32") / 255.0
print(f"Test set loaded: {x_test.shape}")

Dataset URL: https://www.kaggle.com/datasets/datamunge/sign-language-mnist
License(s): CC0-1.0
100% 62.6M/62.6M [00:00<00:00, 179MB/s]

Test set loaded: (7172, 28, 28, 1)


## 2. Load Trained Model

In [11]:
MODEL_PATH = "/content/drive/MyDrive/CV552_SignLanguage/models/cnn_augmented.keras"
model = tf.keras.models.load_model(MODEL_PATH)
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 28, 28, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_6 (Activation)       │ (None, 28, 28, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 14, 14, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_7 (Activation)       │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 7, 7, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 7, 7, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_8 (Activation)       │ (None, 7, 7, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 7, 7, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 6272)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 128)            │       802,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 25)             │         3,225 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,698,317 (10.29 MB)

 Trainable params: 899,289 (3.43 MB)

 Non-trainable params: 448 (1.75 KB)

 Optimizer params: 1,798,580 (6.86 MB)

## 3. Convert to TFLite (Float32)

In [12]:

model.build((None, 28, 28, 1))

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("asl_model.tflite", "wb") as f:
    f.write(tflite_model)
print("Saved: asl_model.tflite")

Saved artifact at '/tmp/tmpjkanwtva'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='input_layer_8')
Output Type:
  TensorSpec(shape=(None, 25), dtype=tf.float32, name=None)
Captures:
  135991919967504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135991919965968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135991919965584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135991919965392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135991919965776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135991919966544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135991919966928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135991919963856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135991919966352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135991919967312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13599191996712

## 4. Convert to TFLite (INT8 Quantized — smaller + faster on Pi)

In [13]:
def representative_dataset():
    for i in range(min(200, len(x_test))):
        yield [x_test[i:i+1]]

converter_int8 = tf.lite.TFLiteConverter.from_keras_model(model)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = representative_dataset
converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_int8.inference_input_type  = tf.float32
converter_int8.inference_output_type = tf.float32

tflite_model_int8 = converter_int8.convert()

with open("asl_model_int8.tflite", "wb") as f:
    f.write(tflite_model_int8)
print("Saved: asl_model_int8.tflite")

Saved artifact at '/tmp/tmpcwqgowa0'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='input_layer_8')
Output Type:
  TensorSpec(shape=(None, 25), dtype=tf.float32, name=None)
Captures:
  135991919967504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135991919965968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135991919965584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135991919965392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135991919965776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135991919966544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135991919966928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135991919963856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135991919966352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135991919967312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13599191996712

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Saved: asl_model_int8.tflite


## 5. Sanity Check — Float32 Model

In [14]:
interpreter = tf.lite.Interpreter(model_path="asl_model.tflite")
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()
print("Input shape  expected:", input_details[0]['shape'])   # [1, 28, 28, 1]
print("Output shape expected:", output_details[0]['shape'])  # [1, 25]

dummy = np.zeros((1, 28, 28, 1), dtype=np.float32)
interpreter.set_tensor(input_details[0]['index'], dummy)
interpreter.invoke()
out = interpreter.get_tensor(output_details[0]['index'])
print("Output (dummy):", out)

Input shape  expected: [ 1 28 28  1]
Output shape expected: [ 1 25]
Output (dummy): [[1.23990327e-03 2.60438723e-03 1.55774966e-01 1.58579890e-02
  1.28998524e-02 3.15655628e-03 7.25502670e-02 6.06083944e-02
  1.76463998e-03 7.48845935e-07 7.30629116e-02 1.98129509e-02
  1.48385676e-04 1.08331465e-03 3.18758637e-01 3.34764854e-03
  1.07022732e-01 1.29500451e-03 8.15173786e-04 2.83932942e-03
  1.72605214e-03 1.23982871e-04 1.06720836e-04 1.42212972e-01
  1.18653697e-03]]


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


## 6. Model Size Comparison

In [15]:
size_keras = os.path.getsize(MODEL_PATH) / 1e6
size_fp32  = os.path.getsize("asl_model.tflite") / 1e6
size_int8  = os.path.getsize("asl_model_int8.tflite") / 1e6

print(f"Model Size Comparison:")
print(f"  Original .keras : {size_keras:.2f} MB")
print(f"  TFLite FP32     : {size_fp32:.2f} MB")
print(f"  TFLite INT8     : {size_int8:.2f} MB")

Model Size Comparison:
  Original .keras : 10.87 MB
  TFLite FP32     : 3.60 MB
  TFLite INT8     : 0.92 MB


## 7. Copy Models to Drive (optional)

In [16]:
import shutil

DRIVE_OUT = "/content/drive/MyDrive/CV552_SignLanguage/tflite"
os.makedirs(DRIVE_OUT, exist_ok=True)

shutil.copy("asl_model.tflite",      os.path.join(DRIVE_OUT, "asl_model.tflite"))
shutil.copy("asl_model_int8.tflite", os.path.join(DRIVE_OUT, "asl_model_int8.tflite"))
print(f"Copied both .tflite files to {DRIVE_OUT}")

Copied both .tflite files to /content/drive/MyDrive/CV552_SignLanguage/tflite
